# Modul B · Kapitel 1.1 — TCREI

## Einen Prompt systematisch entwickeln

**Lernziel:** Du entwickelst einen belastbaren Prompt mit **Task, Context und References**, prüfst die Antwort (**Evaluate**) und verbesserst sie gezielt (**Iterate**).

Wir verwenden durchgehend ein Security-Briefing zu **CVE-2026-3224**. So bleibt der Blick auf der Methode statt auf wechselnden Beispielen.

```
Task + Context + References → Antwort → Evaluate → Iterate
                                      ↑______________|
```

Das Notebook enthält **drei zentrale Challenges**. Führe die Zellen von oben nach unten aus.


---
## 0 · Setup

Das Notebook nutzt die gemeinsame Datei `helfer.py`. Standardmäßig spricht sie ein lokales Ollama-Modell an. Prüfe vor dem Start, dass das konfigurierte Modell verfügbar ist.


In [ ]:
# ▶️ Pakete, Pfade und Helfer laden
import re
import sys
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                 Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

# Beim erneuten Ausführen der Zelle auch Änderungen an helfer.py übernehmen.
import importlib
import helfer
importlib.reload(helfer)
from helfer import BASIS_URL, MODELL, client, frage_dialog, frage_llm, lade_daten, zeige, zeige_vergleich

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")


In [ ]:
# ▶️ Beispieldaten laden
print("Testantwort:", frage_llm("Reply with the single word: ready"))

CVE = lade_daten("cve_datenbank")["CVE-2026-3224"]
BESCHREIBUNG = (
    f"{CVE['id']} ({CVE['title']}). {CVE['summary']} "
    f"Affected: {CVE['component']} {CVE['affected_versions']}. "
    f"Fixed in {CVE['fixed_version']}. CVSS {CVE['cvss']}, {CVE['cwe']}. "
    f"Exploit status: {CVE['exploit_status']}."
)
zeige(BESCHREIBUNG, titel="Ausgangsmaterial")


---
## 1 · Warum TCREI?

Ein Prompt wie *“Explain this CVE”* sagt weder, wer die Antwort braucht, noch wofür. Das Modell muss Ziel, Detailtiefe und Form erraten. TCREI macht diese Entscheidungen sichtbar:

| Schritt | Leitfrage |
|---|---|
| **T — Task** | Was soll für wen in welcher Form entstehen? |
| **C — Context** | Welche konkrete Situation und Entscheidung sind relevant? |
| **R — References** | Auf welche Quellen und Fakten darf sich das Modell stützen? |
| **E — Evaluate** | Welche vorher festgelegten Kriterien erfüllt die Antwort? |
| **I — Iterate** | Was wird gezielt nachgebessert? |

Wichtig: **Evaluate ist kein Bauchgefühl.** Gute Anforderungen sind so konkret, dass wir sie anschließend prüfen können.


In [ ]:
# ▶️ Vergleichsbasis: ein absichtlich vager Prompt
antwort_naiv = frage_llm(f"Explain this CVE.\n\n{BESCHREIBUNG}")
zeige(antwort_naiv, titel="Antwort auf den vagen Prompt")


---
## 2 · T, C und R — den Prompt bauen

Die **Task** enthält hier bereits Rolle, Ergebnis und Format. Besonders das Format ist wichtig: Eine Wortgrenze, feste Überschriften und eine maximale Anzahl von Maßnahmen werden später zu Prüfkriterien.

Der **Context** erklärt nicht das Thema erneut. Er beschreibt die betriebliche Lage und die Entscheidung des Adressaten.

Die **References** begrenzen die Faktenbasis. Eine Quelle ersetzt das Material nicht: Deshalb geben wir zusätzlich die relevanten CVE-Daten mit.


In [ ]:
# ▶️ T — Task: Rolle, Ziel und messbares Ausgabeformat
TASK = """You are a senior vulnerability analyst briefing a CISO.
Create a decision-ready security briefing of at most 180 words.
Use exactly these headings: ## Situation, ## Impact, ## Recommended Actions.
Under Recommended Actions, provide at most three numbered actions.
Name the CVE identifier in the first sentence. Use only the supplied material and references;
if information is missing, state that explicitly rather than inventing it."""


### 🛠️ Challenge 1 — Einen vollständigen TCR-Prompt bauen

Ergänze:

- **Context:** CISO, heutige Entscheidung, zwei produktive Devolutions-Server in Version `2025.3.15.0`, Entra-ID-Authentifizierung und die Bedeutung des Credential Vaults.
- **References:** Verwende die URLs aus `CVE["references"]`.
- **Prompt-Struktur:** Setze `# Task`, `# Context`, `# References` und `# Material` in dieser Reihenfolge zusammen.

Die Überschriften trennen Anweisung, Situation, Quellen und Rohdaten. Das macht den Prompt leichter lesbar und später gezielt veränderbar.


In [ ]:
# 🛠️ Challenge 1: Ergänze Context und References und baue den Prompt.
CONTEXT = ""       # Wer entscheidet was – und in welcher konkreten Lage?
REFERENCES = ""    # Nutze CVE["references"].


def baue_prompt(task, context, references, material):
    """Setzt die TCREI-Eingaben zu klar getrennten Prompt-Abschnitten zusammen."""
    # TODO: Gib einen String mit den Abschnitten # Task, # Context,
    # # References und # Material zurück.
    raise NotImplementedError


In [ ]:
# ✅ Selbsttest Challenge 1
assert len(CONTEXT.split()) >= 25
assert all(begriff in CONTEXT for begriff in ["CISO", "Devolutions Server", "2025.3.15.0"])
assert CVE["id"] in REFERENCES and "https://" in REFERENCES

PROMPT = baue_prompt(TASK, CONTEXT, REFERENCES, BESCHREIBUNG)
koepfe = ["# Task", "# Context", "# References", "# Material"]
assert all(kopf in PROMPT for kopf in koepfe)
assert [PROMPT.index(kopf) for kopf in koepfe] == sorted(PROMPT.index(kopf) for kopf in koepfe)
print("✅ Challenge 1 gelöst")
print(PROMPT[:500], "…")


In [ ]:
# ▶️ Antwort mit TCR erzeugen
briefing = frage_llm(PROMPT)
zeige_vergleich({"Vager Prompt": antwort_naiv, "TCR-Prompt": briefing})


---
## 3 · E — Evaluate

Wir prüfen genau das, was in der Task messbar formuliert wurde:

1. höchstens 180 Wörter,
2. alle drei Überschriften,
3. höchstens drei nummerierte Maßnahmen,
4. die CVE-Kennung.

Diese Prüfung bewertet **Form und Mindestinhalt**, nicht automatisch fachliche Richtigkeit. Dafür bleiben Material und References entscheidend.


In [ ]:
UEBERSCHRIFTEN = ["## Situation", "## Impact", "## Recommended Actions"]

BEISPIEL_GUT = """## Situation
CVE-2026-3224 affects our Devolutions Server deployment.

## Impact
The supplied material describes a critical authentication bypass affecting the credential vault.

## Recommended Actions
1. Upgrade both production servers to the fixed version.
2. Restrict access until the upgrade is complete.
"""

BEISPIEL_SCHLECHT = "No identifier, no required structure."


### 🛠️ Challenge 2 — Anforderungen prüfbar machen

Implementiere `pruefe_briefing(text)`. Das Ergebnis soll die vier booleschen Schlüssel `wortzahl_ok`, `ueberschriften_ok`, `max_drei_massnahmen` und `cve_genannt` enthalten.

Tipp für nummerierte Zeilen: `re.findall(r"(?m)^\s*\d+[.)]\s", text)`.


In [ ]:
# 🛠️ Challenge 2: Übersetze die vier messbaren Anforderungen in Python.
def pruefe_briefing(text):
    """Prüft die messbaren Anforderungen aus TASK."""
    # TODO: Gib ein Dictionary mit vier booleschen Ergebnissen zurück.
    raise NotImplementedError


In [ ]:
# ✅ Selbsttest Challenge 2
erwartet = {"wortzahl_ok", "ueberschriften_ok", "max_drei_massnahmen", "cve_genannt"}
assert set(pruefe_briefing(BEISPIEL_GUT)) == erwartet
assert all(pruefe_briefing(BEISPIEL_GUT).values())
assert pruefe_briefing(BEISPIEL_SCHLECHT)["wortzahl_ok"]
assert not pruefe_briefing(BEISPIEL_SCHLECHT)["ueberschriften_ok"]
assert not pruefe_briefing(BEISPIEL_SCHLECHT)["cve_genannt"]

ERGEBNIS = pruefe_briefing(briefing)
for kriterium, erfuellt in ERGEBNIS.items():
    print(f"{'✔' if erfuellt else '✘'} {kriterium}")


---
## 4 · I — Iterate

Iteration beginnt mit einem **konkreten Prüfergebnis**. Damit das Beispiel reproduzierbar ist, verwenden wir hier eine vorgegebene Modellantwort: Sie erfüllt bewusst nur zwei der vier Kriterien.

Wir schreiben anschließend keinen neuen Prompt von Grund auf. Stattdessen benennen wir nur die beiden festgestellten Mängel und lassen die bereits erfüllten Anforderungen unverändert.


In [ ]:
# ▶️ Vorgegebene Ausgangsantwort für die Iteration
ITERATION_AUSGANGSTEXT = """## Situation
A critical authentication bypass affects the Devolutions Server version used on our two
production systems. An attacker could forge authentication tokens and gain unauthorized
access to credentials stored in the vault.

## Recommended Actions
1. Upgrade both production servers to the fixed version during the next maintenance window.
2. Restrict access to the affected systems until the upgrade is complete.
"""

vorher = pruefe_briefing(ITERATION_AUSGANGSTEXT)
assert sum(vorher.values()) == 2, "Der Ausgangstext soll genau zwei von vier Kriterien erfüllen."

print("Prüfung vor der Iteration:")
for kriterium, erfuellt in vorher.items():
    print(f"{'✔' if erfuellt else '✘'} {kriterium}")


### 🛠️ Challenge 3 — Genau die erkannten Mängel beheben

Die Prüfung zeigt:

- Die Überschrift `## Impact` fehlt.
- Die Kennung `CVE-2026-3224` fehlt.

Formuliere eine kurze englische Nachbesserung, die genau diese beiden Punkte korrigiert. Fordere außerdem, dass die bereits erfüllte Wortgrenze und die maximale Anzahl von drei Maßnahmen erhalten bleiben. Das Modell darf keine neuen Fakten erfinden.

Beziehe dich auf *your previous briefing*, statt die ursprüngliche Aufgabe zu wiederholen.


In [ ]:
# 🛠️ Challenge 3: Formuliere die gezielte Nachbesserung.
NACHBESSERUNG = ""

if not NACHBESSERUNG:
    raise NotImplementedError("Formuliere eine konkrete Nachbesserung.")


In [ ]:
# ▶️ Nachbessern und mit denselben vier Kriterien erneut prüfen
verlauf = [
    {"role": "user", "content": PROMPT},
    {"role": "assistant", "content": ITERATION_AUSGANGSTEXT},
    {"role": "user", "content": NACHBESSERUNG},
]
ITERATION_ERGEBNIS = frage_dialog(verlauf)
nachher = pruefe_briefing(ITERATION_ERGEBNIS)

zeige_vergleich({"Vor Iteration": ITERATION_AUSGANGSTEXT,
                 "Nach Iteration": ITERATION_ERGEBNIS})

print("Prüfung nach der Iteration:")
for kriterium in vorher:
    print(f"{kriterium:<24} vorher: {'✔' if vorher[kriterium] else '✘'}  "
          f"nachher: {'✔' if nachher[kriterium] else '✘'}")


---
## 5 · Fazit

Ein guter TCREI-Durchlauf ist kurz und nachvollziehbar:

1. **Task** legt Ziel, Adressat und prüfbares Format fest.
2. **Context** liefert die konkrete Entscheidungssituation.
3. **References** und Material begrenzen die Faktenbasis.
4. **Evaluate** prüft die zuvor definierten Kriterien.
5. **Iterate** korrigiert nur die erkannten Schwächen.

Der wichtigste Zusammenhang: **Was du später prüfen willst, musst du vorher konkret verlangen.**

### Transferfrage

Welche der vier automatischen Prüfungen sagt noch nichts über die fachliche Qualität aus? Ergänze gedanklich ein fünftes Kriterium, das in deinem eigenen Arbeitskontext wirklich entscheidend wäre.
